In [ ]:
def camel_case_to_snake_case(name):
    return re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()

class EnumMapping:
    def __init__(self, df: pd.DataFrame = pd.DataFrame(), process_config: Dict[str, Dict[str, Any]] = None):
        self.df = df
        self.process_config = process_config

    def check_null(self, column_name: str = "") -> pd.Series:
        """
        Check each item of pd.Series is null/na or not

        Returns:
            pd.Series: a mask of pd.Series, contains True if item is null/na, otherwise False
        """

        return (self.df[column_name] == "") | (self.df[column_name].isna())

    def map_enum(self, column_name, enum_data):
        """
        Map enum data to pd.Series.

        Args:
            column_name (str): column need map enum
            enum_data (Dict): data to map enum
        """
        map_enum_mask = ~self.check_null(column_name)

        self.df.loc[map_enum_mask, column_name] = self.df.loc[map_enum_mask, column_name].map(lambda x: enum_data.get(x, ""))

    def process_data(self) -> pd.DataFrame:
        """
        Process data with process_config.

        Returns:
            pd.DataFrame: a pd.DataFrame, contains processed data
        """
        for column_name, enum_data in self.process_config.items():
            if column_name not in self.df.columns:
                raise ValueError(f"{self.process_data.__name__}: {column_name} not in dataframe columns")
            if not enum_data:
                raise ValueError(f"{self.process_data.__name__}: Can not found enum data")

            self.map_enum(column_name, enum_data)

        return self.df

class DataValidation:
    def __init__(self, df: pd.DataFrame, validation_config: Dict[str, Dict[str, Any]], error_messages: Dict = None, mapped_columns: Dict = None):
        self.df = df
        self.validation_config = validation_config
        if "result_detail" not in self.df.columns:
            self.df["result_detail"] = [set() for _ in range(self.df.shape[0])]
        self.validation_action_mapping = {
            "mandatory": self.check_mandatory,
            "datetime_format": self.check_datetime_format,
            "error_value": self.check_error_value,
            "refer_required": self.check_refer_required,
            "refer_value": self.check_refer_value,
        }
        self.error_messages = error_messages
        self.mapped_columns = mapped_columns

    def check_null(self, column_name: str = "") -> pd.Series:
        """Check each item of pd.Series is null/na or not

        Returns:
            pd.Series: a mask of pd.Series, contains True if item is null/na, otherwise False
        """

        return (self.df[column_name] == "") | (self.df[column_name].isna())

    def check_mandatory(self, **kwargs):
        """Check required column has empty value or not. If empty, add error message to result_detail"""

        column_name: str = kwargs.get("column_name")
        is_mandatory: bool = kwargs.get("is_mandatory")
        query_string: str = kwargs.get("query_string")

        message = self.error_messages["required"].format(self.mapped_columns.get(column_name, column_name))
        mandatory_mask = self.check_null(column_name)

        if not is_mandatory:
            raise ValueError(f"[{self.check_mandatory.__name__}: {column_name}] Field 'is_mandatory' is required")

        if is_mandatory:
            if query_string:
                query_index = self.df.query(query_string).index
                query_mask = self.df.index.isin(query_index)
                mandatory_mask = query_mask & mandatory_mask

            self.df.loc[mandatory_mask, "result_detail"] = self.df.loc[mandatory_mask, "result_detail"].map(lambda x: x.union({message}))

    def check_datetime_format(self, **kwargs):
        """Check value of column is datetime or not. If not, add error message to result_detail"""

        column_name: str = kwargs.get("column_name")
        date_format: str = kwargs.get("format")
        query_string: str = kwargs.get("query_string")

        message = self.error_messages["datetime_format"].format(self.mapped_columns.get(column_name, column_name))
        datetime_mask = (~self.check_null(column_name)) & (pd.to_datetime(self.df[column_name], format=date_format, errors="coerce").isna())

        if not date_format:
            raise ValueError(f"[{self.check_datetime_format.__name__}: {column_name}] Date_format is required")

        if query_string:
            query_index = self.df.query(query_string).index
            query_mask = self.df.index.isin(query_index)
            datetime_mask = datetime_mask & query_mask

        self.df.loc[datetime_mask, "result_detail"] = self.df.loc[datetime_mask, "result_detail"].map(lambda x: x.union({message}))

    def check_error_value(self, **kwargs):
        """Check value of column is in value_list or not. If not, add error message to result_detail"""

        column_name: str = kwargs.get("column_name")
        value_list: List = kwargs.get("value_list")
        query_string: str = kwargs.get("query_string")

        if not value_list:
            raise ValueError(f"[{self.check_error_value.__name__}: {column_name}] Value_list is required")

        message = self.error_messages["error_value"].format(self.mapped_columns.get(column_name, column_name))
        value_mask = (~self.check_null(column_name)) & (~self.df[column_name].isin(value_list))

        if query_string:
            query_index = self.df.query(query_string).index
            query_mask = self.df.index.isin(query_index)
            value_mask = value_mask & query_mask

        self.df.loc[value_mask, "result_detail"] = self.df.loc[value_mask, "result_detail"].apply(lambda x: x.union({message}))

    def check_refer_required(self, **kwargs):
        """Check refer column has empty value or not. If empty, add error message to result_detail"""

        column_name: str = kwargs.get("column_name")
        refer_info: Dict = kwargs.get("refer_info")

        if not refer_info:
            raise ValueError(f"[{self.check_refer_required.__name__}: {column_name}] Must have refer info")

        for refer_column, refer_data in refer_info.items():
            refer_value: Any = refer_data.get("refer_value")
            query_string: str = refer_data.get("query_string")

            if not refer_column in self.df.columns:
                raise ValueError(f"[{self.check_refer_required.__name__}: {column_name}] {refer_column} not in dataframe columns.")

            if not refer_value:
                raise ValueError(f"[{self.check_refer_required.__name__}: {column_name}] {refer_column} must have refer value")

            if not query_string:
                raise ValueError(f"[{self.check_refer_required.__name__}: {column_name}] {refer_column} must have query string")

            query_index = self.df.query(query_string).index
            query_mask = self.df.index.isin(query_index)
            refer_mask = self.check_null(column_name) & query_mask

            message = self.error_messages["refer_required"].format(self.mapped_columns.get(column_name, column_name), self.mapped_columns.get(refer_column, refer_column), refer_value)

            self.df.loc[refer_mask, "result_detail"] = self.df.loc[refer_mask, "result_detail"].map(lambda x: x.union({message}))

    def check_refer_value(self, **kwargs):
        """Check refer column has empty value or not. If empty, add error message to result_detail"""

        column_name: str = kwargs.get("column_name")
        refer_info: Dict = kwargs.get("refer_info")

        if not refer_info:
            raise ValueError(f"[{self.check_refer_value.__name__}: {column_name}] Must have refer value")

        for refer_column, refer_data in refer_info.items():
            if refer_column not in self.df.columns:
                raise ValueError(f"[{self.check_refer_value.__name__}: {column_name}] {refer_column} is not in dataframe columns")

            column_value: Any = refer_data.get("column_value")
            refer_value: Any = refer_data.get("refer_value")
            query_string: str = refer_data.get("query_string")

            if not column_value:
                raise ValueError(f"[{self.check_refer_value.__name__}: {column_name}] Must have column value")
            if not refer_value:
                raise ValueError(f"[{self.check_refer_value.__name__}: {column_name}] Must have refer value")
            if not query_string:
                raise ValueError(f"[{self.check_refer_value.__name__}: {column_name}] Must have query string")

            query_index = self.df.query(query_string).index
            query_mask = self.df.index.isin(query_index)
            refer_mask = ~self.check_null(column_name) & query_mask

            message = self.error_messages["refer_value"].format(self.mapped_columns.get(column_name, column_name), column_value, self.mapped_columns.get(refer_column, refer_column), refer_value)

            self.df.loc[refer_mask, "result_detail"] = self.df.loc[refer_mask, "result_detail"].map(lambda x: x.union({message}))

    def run(self):
        """Run all validations across all configured columns"""

        for column_name, column_validation_config in self.validation_config.items():
            if not column_name:
                raise ValueError("column_name is required")
            if column_name not in self.df.columns:
                raise ValueError(f"{column_name} is not in dataframe")

            for action, config in column_validation_config.items():
                func = self.validation_action_mapping.get(action, None)
                if func:
                    func(column_name=column_name, **config)
                else:
                    raise ValueError(f"Invalid action: {action}")

        return self.df

def process_data(df: pd.DataFrame = pd.DataFrame(), mapped_columns: Dict = None):
    """
    Process data, like validate and map enum before transfer and send data

    Args:
        df (pd.DataFrame): input data
        mapped_columns (Dict): dict of normalized columns as keys and origin columns as values

    Returns:
        pd.DataFrame: data after processing.
    """

    LEARNER_DATA_ENUM = LEARNER_PREDATA["predata"]["learner_account"]["enum"]

    COUNTRY_CODE_ENUM = {item["name"]: item["id"] for item in LEARNER_DATA_ENUM["CountryCode"]}
    NATIONALITY_ENUM = {item["name"]: item["id"] for item in LEARNER_DATA_ENUM["NationalityCode"]}
    RACE_ENUM = {item["name"]: str(item["id"]) for item in LEARNER_DATA_ENUM["RaceCode"]}

    GENDER_DATA = [k for k in LEARNER_DATA_ENUM["Gender"].keys()]
    RESIDENTIAL_STATUS_DATA = [k for k in LEARNER_DATA_ENUM["ResidentialStatus"].keys()]
    PASS_TYPE_DATA = [k for k in LEARNER_DATA_ENUM["PassType"].keys()]
    RESIDENTIAL_ADDRESS_DATA = [k for k in LEARNER_DATA_ENUM["ResidentialAddress"].keys()]
    NO_FLOOR_AND_UNIT_DATA = [k for k in LEARNER_DATA_ENUM["noFloorAndUnit"].keys()]
    IS_EMPLOYED_DATA = [k for k in LEARNER_DATA_ENUM["IsEmployed"].keys()]
    VIP_DATA = [k for k in LEARNER_DATA_ENUM["VIP"].keys()]
    COUNTRY_CODE_DATA = [item["name"] for item in LEARNER_DATA_ENUM["CountryCode"]]
    NATIONALITY_DATA = [item["name"] for item in LEARNER_DATA_ENUM["NationalityCode"]]
    RACE_DATA = [item["name"] for item in LEARNER_DATA_ENUM["RaceCode"]]

    error_messages = {
        "required": "[{}] Required field.",
        "error_value": "[{}] Value not in dropdown list.",
        "refer_required": "[{}] Required if '{}' is '{}'.",
        "refer_value": "[{}] Must be '{}' if '{}' is '{}'.",
        "datetime_format": "[{}] Datetime format must be YYYY-MM-DD.",
    }

    DATE_FORMAT = "%Y-%m-%d"

    data_validation_config = {
        "user_id": {
            "mandatory": {
                "is_mandatory": True,
                "query_string": None
            }
        },
        "name": {
            "mandatory": {
                "is_mandatory": True,
                "query_string": None
            }
        },
        "gender": {
            "mandatory": {
                "is_mandatory": True,
                "query_string": None
            },
            "error_value": {
                "value_list": GENDER_DATA,
                "query_string": None
            }
        },
        "date_of_birth": {
            "mandatory": {
                "is_mandatory": True,
                "query_string": None
            },
            "datetime_format": {
                "format": DATE_FORMAT,
                "query_string": None
            }
        },
        "residential_status": {
            "mandatory": {
                "is_mandatory": True,
                "query_string": None
            },
            "error_value": {
                "value_list": RESIDENTIAL_STATUS_DATA,
                "query_string": None
            }
        },
        "nric_fin": {
            "mandatory": {
                "is_mandatory": True,
                "query_string": None
            }
        },
        "pass_type": {
            "error_value": {
                "value_list": PASS_TYPE_DATA,
                "query_string": None
            },
            "refer_required": {
                "refer_info": {
                    "residential_status": {
                        "refer_value": RESIDENTIAL_STATUS_DATA[-1],
                        "query_string": f"residential_status == '{RESIDENTIAL_STATUS_DATA[-1]}'",
                    }
                }
            }
        },
        "pass_expiry_date": {
            "datetime_format": {
                "format": DATE_FORMAT,
                "query_string": f"residential_status == '{RESIDENTIAL_STATUS_DATA[-1]}'"
            }
        },
        "work_permit_id":{
            "refer_required": {
                "refer_info": {
                    "pass_type": {
                        "refer_value": PASS_TYPE_DATA[1],
                        "query_string": f"pass_type == '{PASS_TYPE_DATA[1]}'",
                    }
                }
            }
        },
        "race": {
            "mandatory": {
                "is_mandatory": True,
                "query_string": None
            },
            "error_value": {
                "value_list": RACE_DATA,
                "query_string": None
            }
        },
        "country_region_of_birth": {
            "mandatory": {
                "is_mandatory": True,
                "query_string": None
            },
            "error_value": {
                "value_list": COUNTRY_CODE_DATA,
                "query_string": None
            }
        },
        "nationality": {
            "mandatory": {
                 "is_mandatory": True,
                "query_string": None
            },
            "error_value": {
                "value_list": NATIONALITY_DATA,
                "query_string": None
            },
            "refer_value": {
                "refer_info": {
                    "residential_status": {
                        "column_value": "SINGAPORE CITIZEN",
                        "refer_value": RESIDENTIAL_STATUS_DATA[0],
                        "query_string": f"residential_status == '{RESIDENTIAL_STATUS_DATA[0]}' and nationality != 'SINGAPORE CITIZEN'"
                    }
                }
            }
        },
        "residential_address": {
            "mandatory": {
                 "is_mandatory": True,
                "query_string": None
            },
            "error_value": {
                "value_list": RESIDENTIAL_ADDRESS_DATA,
                "query_string": None
            }
        },
        "country_or_region": {
            "mandatory": {
                 "is_mandatory": True,
                "query_string": None
            },
            "error_value": {
                "value_list": COUNTRY_CODE_DATA,
                "query_string": None
            },
            "refer_value": {
                "refer_info": {
                    "residential_address": {
                        "column_value": "SINGAPORE",
                        "refer_value": RESIDENTIAL_ADDRESS_DATA[0],
                        "query_string": f"residential_address == '{RESIDENTIAL_ADDRESS_DATA[0]}' and country_or_region != 'SINGAPORE'"
                    }
                }
            }
        },
        "postal_code": {
            "refer_required": {
                "refer_info": {
                    "residential_address": {
                        "refer_value": RESIDENTIAL_ADDRESS_DATA[0],
                        "query_string": f"residential_address == '{RESIDENTIAL_ADDRESS_DATA[0]}'",
                    }
                }
            }
        },
        "block_building_no": {
            "refer_required": {
                "refer_info": {
                    "residential_address": {
                        "refer_value": RESIDENTIAL_ADDRESS_DATA[0],
                        "query_string": f"residential_address == '{RESIDENTIAL_ADDRESS_DATA[0]}'",
                    }
                }
            }
        },
        "street_name": {
            "refer_required": {
                "refer_info": {
                    "residential_address": {
                        "refer_value": RESIDENTIAL_ADDRESS_DATA[0],
                        "query_string": f"residential_address == '{RESIDENTIAL_ADDRESS_DATA[0]}'",
                    }
                }
            }
        },
        "floor_unit_number_is_not_applicable": {
            "error_value": {
                "value_list": NO_FLOOR_AND_UNIT_DATA,
                "query_string": None
            },
            "refer_required": {
                "refer_info": {
                    "residential_address": {
                        "refer_value": RESIDENTIAL_ADDRESS_DATA[0],
                        "query_string": f"residential_address == '{RESIDENTIAL_ADDRESS_DATA[0]}'",
                    }
                }
            }
        },
        "floor_number": {
            "refer_required": {
                "refer_info": {
                    "floor_unit_number_is_not_applicable": {
                        "refer_value": NO_FLOOR_AND_UNIT_DATA[1],
                        "query_string": f"residential_address == '{RESIDENTIAL_ADDRESS_DATA[0]}' and floor_unit_number_is_not_applicable == '{NO_FLOOR_AND_UNIT_DATA[1]}'",
                    }
                }
            }
        },
        "unit_number": {
            "refer_required": {
                "refer_info": {
                    "floor_unit_number_is_not_applicable": {
                        "refer_value": NO_FLOOR_AND_UNIT_DATA[1],
                        "query_string": f"residential_address == '{RESIDENTIAL_ADDRESS_DATA[0]}' and floor_unit_number_is_not_applicable == '{NO_FLOOR_AND_UNIT_DATA[1]}'",
                    }
                }
            }
        },
        "address_line_1": {
            "refer_required": {
                "refer_info": {
                    "residential_address": {
                        "refer_value": RESIDENTIAL_ADDRESS_DATA[1],
                        "query_string": f"residential_address == '{RESIDENTIAL_ADDRESS_DATA[1]}'",
                    }
                }
            }
        },
        "address_line_2": {
            "refer_required": {
                "refer_info": {
                    "residential_address": {
                        "refer_value": RESIDENTIAL_ADDRESS_DATA[1],
                        "query_string": f"residential_address == '{RESIDENTIAL_ADDRESS_DATA[1]}'",
                    }
                }
            }
        },
        "employed": {
            "mandatory": {
                 "is_mandatory": True,
                "query_string": None
            },
            "error_value": {
                "value_list": IS_EMPLOYED_DATA,
                "query_string": None
            }
        },
        "vip": {
            "mandatory": {
                 "is_mandatory": True,
                "query_string": None
            },
            "error_value": {
                "value_list": VIP_DATA,
                "query_string": None
            }
        }
    }

    map_enum_config = {
        "gender": LEARNER_DATA_ENUM["Gender"],
        "residential_status": LEARNER_DATA_ENUM["ResidentialStatus"],
        "pass_type": LEARNER_DATA_ENUM["PassType"],
        "race": RACE_ENUM,
        "country_region_of_birth": COUNTRY_CODE_ENUM,
        "nationality": NATIONALITY_ENUM,
        "residential_address": LEARNER_DATA_ENUM["ResidentialAddress"],
        "country_or_region": COUNTRY_CODE_ENUM,
        "floor_unit_number_is_not_applicable": LEARNER_DATA_ENUM["noFloorAndUnit"],
        "employed": LEARNER_DATA_ENUM["IsEmployed"],
        "vip": LEARNER_DATA_ENUM["VIP"]
    }

    if df.empty:
        raise ValueError(f"[{process_data.__name__}] Dataframe can not be empty")

    # Fill default for pass type if residential status is not Foreigner and pass type is empty
    fill_pass_type_default_value_mask = (df["pass_type"] == "") & (df["residential_status"] != RESIDENTIAL_STATUS_DATA[-1])
    df.loc[fill_pass_type_default_value_mask, "pass_type"] = df.loc[fill_pass_type_default_value_mask, "pass_type"].map(lambda _: "None")

    df["result_detail"] = [set() for _ in range(df.shape[0])]

    # Validate data
    df = DataValidation(
        df=df,
        validation_config=data_validation_config,
        error_messages=error_messages,
        mapped_columns=mapped_columns
    ).run()

    # Map enum
    df = EnumMapping(df, map_enum_config).process_data()

    df["result"] = df["result_detail"].map(lambda x: "Pass" if len(x) == 0 else "Fail")
    return df

@pytest.fixture(scope="session")
def Sourcesetup():
    returndata = {}

    edit_template = LEARNER_PREDATA["predata"]["learner_account"]["template"][ "edit_template"]

    information_list = sysadmin.AWB.get_information_list_for_learner_profile()
    marketing_consent_data = [
        {
            "id": item.get("id"),
            "description": item.get("description"),
            "allowMarketing": True,
            "sms": True,
            "email": True,
            "call": True,
        }
        for item in information_list.get("MarketingConsent", [])
    ]

    folder_template_name = "learner_account"
    file_name = "API learner profile template.xlsx"
    template_path = os.path.join(BASE_PATH, "data", "template", folder_template_name)
    file_path = os.path.join(template_path, file_name)

    if not os.path.exists(file_path):
        logger.error("File template not found")
        return returndata
    else:
        logger.info("File template found")
        returndata["file_path"] = file_path

    df_learner_profile = pd.read_excel(file_path)

    origin_columns = deepcopy(df_learner_profile.columns.tolist())
    df_learner_profile.columns = df_learner_profile.columns.str.lower().map(lambda x: re.sub("[^a-zA-Z0-9]", " ", x)).str.strip().str.replace(" ", "_")

    mapped_columns = dict(zip(df_learner_profile.columns.tolist(), origin_columns))

    df_learner_profile[["postal_code", "floor_number", "unit_number"]] = df_learner_profile[["postal_code", "floor_number", "unit_number"]].fillna(np.nan).astype("Int64")

    df_learner_profile = df_learner_profile.astype(str).fillna("").replace(["nan", "<NA>"], "").apply(lambda x: x.str.strip(), axis=1)

    df_learner_profile_result = df_learner_profile.copy(deep=True)
    df_learner_profile_result.columns = origin_columns

    df_learner_profile = process_data(df_learner_profile, mapped_columns)

    for i in range(df_learner_profile.shape[0]):
        if df_learner_profile.loc[i, "result"] == "Fail":
            continue

        request_body = deepcopy(edit_template)

        request_body["userId"] = df_learner_profile.loc[i, "user_id"]
        search_learner_account_result = sysadmin.AWB.search_learner_account_by_user_id(str(request_body["userId"]))
        if search_learner_account_result["data"] is not None:
            request_body["id"] = search_learner_account_result["data"]["id"]
        else:
            df_learner_profile.loc[i, "result"] = "Fail"
            df_learner_profile.loc[i, "result_detail"].add(search_learner_account_result[ "error_message"])
            continue

        unhidden_info_result = sysadmin.AWB.get_unhidden_learner_account_info_by_id(request_body["id"])
        if unhidden_info_result["data"] is not None:
            for key, value in unhidden_info_result["data"].items():
                request_body[key] = value
        else:
            df_learner_profile.loc[i, "result"] = "Fail"
            df_learner_profile.loc[i, "result_detail"].add(unhidden_info_result["error_message"])
            continue

        # personalInformation
        request_body["personalInformation"]["name"] = df_learner_profile.loc[i, "name"]
        request_body["personalInformation"]["gender"] = df_learner_profile.loc[i, "gender"]
        request_body["personalInformation"]["residentialStatus"] = df_learner_profile.loc[i, "residential_status"]
        request_body["personalInformation"]["race"] = df_learner_profile.loc[i, "race"]
        request_body["personalInformation"]["countryRegionOfBirth"] = df_learner_profile.loc[i, "country_region_of_birth"]
        request_body["personalInformation"]["nationality"] = df_learner_profile.loc[i, "nationality"]
        request_body["personalInformation"]["employeeId"] = df_learner_profile.loc[i, "employee_id"]
        request_body["personalInformation"]["passType"] = df_learner_profile.loc[i, "pass_type"]
        request_body["personalInformation"]["passExpiryDate"] = df_learner_profile.loc[i, "pass_expiry_date"]
        request_body["personalInformation"]["workPermitId"] = df_learner_profile.loc[i, "work_permit_id"]

        request_body["dateOfBirth"] = df_learner_profile.loc[i, "date_of_birth"]
        request_body["idNumber"] = df_learner_profile.loc[i, "nric_fin"]

        # addressInformation - residentialAddress
        request_body["addressInformation"]["residentialAddress"]["addressType"] = df_learner_profile.loc[i, "residential_address"]
        request_body["addressInformation"]["residentialAddress"]["postalCode"] = str(df_learner_profile.loc[i, "postal_code"])
        request_body["addressInformation"]["residentialAddress"]["blockBuildingNo"] = str(df_learner_profile.loc[i, "block_building_no"])
        request_body["addressInformation"]["residentialAddress"]["buildingName"] = df_learner_profile.loc[i, "building_name"]
        request_body["addressInformation"]["residentialAddress"]["streetName"] = df_learner_profile.loc[i, "street_name"]
        request_body["addressInformation"]["residentialAddress"]["floorNumber"] = str(df_learner_profile.loc[i, "floor_number"])
        request_body["addressInformation"]["residentialAddress"]["unitNumber"] = str(df_learner_profile.loc[i, "unit_number"])
        request_body["addressInformation"]["residentialAddress"]["addressLine1"] = df_learner_profile.loc[i, "address_line_1"]
        request_body["addressInformation"]["residentialAddress"]["addressLine2"] = df_learner_profile.loc[i, "address_line_2"]
        request_body["addressInformation"]["residentialAddress"]["noFloorAndUnit"] = bool(df_learner_profile.loc[i, "floor_unit_number_is_not_applicable"])
        request_body["addressInformation"]["residentialAddress"]["countryOrRegion"] = df_learner_profile.loc[i, "country_or_region"]

        # employmentInformation
        request_body["employmentInformation"]["isEmployed"] = bool(df_learner_profile.loc[i, "employed"])

        # vip
        request_body["vip"] = bool(df_learner_profile.loc[i, "vip"])

        # declarations
        request_body["declarations"] = marketing_consent_data

        logger.info(f"Request body: {request_body}")

        edit_learner_profile_result = sysadmin.AWB.edit_learner_profile(request_body)
        df_learner_profile.loc[i, "result"] = edit_learner_profile_result["result"]
        if edit_learner_profile_result["result"] == "Fail":
            df_learner_profile.loc[i, "result_detail"].add(edit_learner_profile_result["error_message"])

    df_learner_profile["result_detail"] = df_learner_profile["result_detail"].map(lambda x: "\n".join(x))

    df_learner_profile_result[df_learner_profile_result.columns[-2:]] = df_learner_profile[["result", "result_detail"]].fillna("")

    result_file_name = "API edit learner profile result {}.xlsx".format(datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
    result_file_path = os.path.join(BASE_PATH, "data", "template", "learner_account_result", result_file_name)
    with pd.ExcelWriter(result_file_path) as writer:
        df_learner_profile_result.to_excel(writer, sheet_name="Edit learner profile result", index=False)

    return returndata